In [0]:
import logging
import random
import re
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("week2")

# ── Plot defaults ──────────────────────────────────────────────────────────────
plt.rcParams["figure.dpi"]        = 110
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# ── Catalog / schema constants ─────────────────────────────────────────────────
CATALOG = "sandbox_catalog"
SCHEMA  = "banking_details"

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

In [0]:
# Create the schema (database) inside our catalog if it does not already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")



In [0]:
#Customer table
PROVINCE = [
    "Alberta", "AB", "alberta",
    "British Columbia", "BC", "british columbia",
    "Manitoba", "MB", "manitoba",
    "New Brunswick", "NB", "new brunswick",
    "Newfoundland and Labrador", "NL", "newfoundland and labrador",
    "Nova Scotia", "NS", "nova scotia",
    "Ontario", "ON", "ontario",
    "Prince Edward Island", "PE", "prince edward island",
    "Quebec", "QC", "quebec",
    "Saskatchewan", "SK", "saskatchewan"
]

customers_data = []
for i in range(1000):
    join_date = (datetime.now() - timedelta(days=random.randint(0, 365*10))).date()
    customers_data.append((i, random.randint(18, 70), random.choice(PROVINCE), random.choice(["$0-$25K", "$25K-$50K", "$50K-$75K", "$75K-$100K", "$100K-$150K", "$150K-$200K", "$200K+"]), join_date))

spark.createDataFrame(customers_data, ["customer_id", "age", "province", "income_bracket", "join_date"]).write.mode("overwrite").saveAsTable("customers")
display(spark.sql("select * from customers LIMIT 5"))

In [0]:
#Accounts table
ACCOUNT_TYPE= ['savings', 'CHECKINGS', 'Savings', 'Checkings']
STATUS = ['active', 'inactive', 'closed']
STATUS_WEIGHTS = [70, 20, 10]
CUSTOMER_IDS = list(range(1000))
START_DATE = (datetime.now() - timedelta(days=365*10)).date()
END_DATE = datetime.now().date()

accounts_data= []

for i in range(1500):
    customer_id = random.choice(CUSTOMER_IDS)
    account_type = random.choice(ACCOUNT_TYPE)
    random_days = random.randint(0, (END_DATE - START_DATE).days)
    open_date = (START_DATE + timedelta(days=random_days))

    if random.random() < 0.05:
        balance = str(round(random.uniform(-5000, -1), 2))
    else:
        balance = str(round(random.uniform(100, 500000), 2))

    accounts_data.append((i, customer_id, random.choice(ACCOUNT_TYPE),random.choice(STATUS), join_date, random.randint(1000, 1000000)))

spark.createDataFrame(accounts_data, ["account_id","customer_id", "account_type", "status", "open_date", "balance"]).write.mode("overwrite").saveAsTable("accounts")
display(spark.sql("select * from accounts LIMIT 5"))


In [0]:
# Transactions table
TRANSACTION_TYPE = ["deposit", "withdrawal", "transfer"]
TRANSACTION_WEIGHTS = [30, 50, 20]  

MERCHANT_CATEGORIES = ["Groceries", "Dining", "Gas & Transport", 
                        "Healthcare", "Entertainment", "Utilities", 
                        "Online Shopping", "Travel", "ATM Withdrawal"]

ACCOUNT_IDS = list(range(0, 1000))

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

transactions_data = []

for i in range(10000): 
    account_id = random.choice(ACCOUNT_IDS)

    transaction_type = random.choices(TRANSACTION_TYPE, weights=TRANSACTION_WEIGHTS)[0]

    random_days = random.randint(0, (END_DATE - START_DATE).days)
    transaction_date = (START_DATE + timedelta(days=random_days)).strftime("%Y-%m-%d")

    if random.random() < 0.03:
        amount = None
    elif transaction_type == "withdrawal" and random.random() < 0.05:
        amount = str(round(random.uniform(-500, -1), 2))
    else:
        amount = str(round(random.uniform(1, 5000), 2))

    if transaction_type in ("deposit", "transfer"):
        merchant_category = None
    elif random.random() < 0.05:
        merchant_category = None
    else:
        merchant_category = random.choice(MERCHANT_CATEGORIES)

    transactions_data.append((
        i, account_id, transaction_type, transaction_date, amount, merchant_category
    ))

spark.createDataFrame(
    transactions_data, 
    ["transaction_id", "account_id", "transaction_type", "transaction_date", "amount", "merchant_category"]
).write.mode("overwrite").saveAsTable("transactions")

display(spark.sql("select * from transactions LIMIT 5"))

In [0]:
# Loan Applications table
LOAN_TYPE = ["mortgage", "auto", "personal"]
LOAN_STATUS = ["approved", "denied", "pending"]
STATUS_WEIGHTS = [50, 35, 15]

LOAN_AMOUNT_RANGES = {
    "mortgage": (100000, 900000),
    "auto": (5000, 80000),
    "personal": (1000, 50000)
}

INTEREST_RATE_RANGES = {
    "mortgage": (3.0, 7.0),
    "auto": (4.0, 12.0),
    "personal": (6.0, 25.0)
}

CUSTOMER_IDS = list(range(0, 1000))

START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2024, 12, 31)

loan_data = []

for i in range(600):
    customer_id = random.choice(CUSTOMER_IDS)
    loan_type = random.choice(LOAN_TYPE)

    random_days = random.randint(0, (END_DATE - START_DATE).days)
    application_date = (START_DATE + timedelta(days=random_days)).strftime("%Y-%m-%d")

    if random.random() < 0.05:
        amount_requested = None
    else:
        low, high = LOAN_AMOUNT_RANGES[loan_type]
        amount_requested = round(random.uniform(low, high), 2)

    # Status
    status = random.choices(LOAN_STATUS, weights=STATUS_WEIGHTS)[0]

    # Interest rate 
    if random.random() < 0.03:
        interest_rate = round(random.uniform(100.0, 200.0), 2) 
    else:
        low, high = INTEREST_RATE_RANGES[loan_type]
        interest_rate = round(random.uniform(low, high), 2)

    loan_data.append((
        i, customer_id, loan_type, amount_requested, status, application_date, interest_rate
    ))

spark.createDataFrame(
    loan_data,
    ["application_id", "customer_id", "loan_type", "amount_requested", "status", "application_date", "interest_rate"]
).write.mode("overwrite").saveAsTable("loan_applications")

display(spark.sql("select * from loan_applications LIMIT 5"))